## 🎯 Learning Objectives
* Understand the necessity and benefits of creating custom agents in AutoGen beyond the default `AssistantAgent` and `UserProxyAgent`.
* Learn how to define a custom agent class by inheriting from `autogen.ConversableAgent` and overriding the `generate_reply` method.
* Implement explicit message routing within custom agents using the `to` field in the reply dictionary.
* Identify common use cases and performance considerations for custom agents and advanced message routing strategies.


## Custom Agents and Advanced Message Routing in AutoGen

AutoGen provides powerful default agents like `AssistantAgent` and `UserProxyAgent` that cover a wide range of collaborative AI tasks. However, real-world applications often demand highly specialized behaviors, integrations with external systems, or complex decision-making processes that go beyond what these general-purpose agents can offer. This is where **custom agents** become indispensable.

### Why Custom Agents?

Imagine AutoGen's built-in agents as off-the-shelf tools – a hammer for general construction, a screwdriver for fasteners. They're incredibly useful for many tasks. But what if you need to perform micro-soldering on a circuit board, or precisely assemble a complex engine? You'd need specialized tools, perhaps even custom-designed robotic arms. Custom agents in AutoGen are precisely these specialized tools.

You'd create a custom agent when you need to:

1.  **Integrate with External Systems:** Connect to databases, APIs (e.g., Google AI Studio, Hugging Face models, internal enterprise tools), web scrapers, or IoT devices.
2.  **Implement Domain-Specific Logic:** Embed business rules, complex algorithms, or specific data processing steps that an LLM might struggle with or perform inefficiently.
3.  **Manage State and Memory:** Maintain persistent information across conversations or implement sophisticated memory retrieval mechanisms.
4.  **Control Conversation Flow Explicitly:** Dictate precisely which agent should receive the next message based on the current message content, agent state, or external conditions.
5.  **Optimize Resource Usage:** Ensure that expensive LLM calls or API requests are only made when absolutely necessary, by having a custom agent perform pre-processing or conditional logic.

### The Anatomy of a Custom Agent

At its core, a custom agent in AutoGen typically inherits from `autogen.ConversableAgent`. The most critical method to override is `generate_reply`. This method is where your agent's custom logic resides. It receives the conversation history (`messages`) and the `sender` of the last message, and it's responsible for deciding what to do next.

### Advanced Message Routing

Once you have specialized agents, the next challenge is ensuring messages flow correctly between them. This is **message routing**. Think of it like a sophisticated postal service or a switchboard operator for your agents. Instead of messages just going back to the sender or following a simple round-robin, routing allows you to direct messages to the *most appropriate* agent for the next step.

AutoGen offers several ways to achieve routing:

1.  **Explicit `to` Field in Reply:** A custom agent's `generate_reply` method can return a dictionary containing a `"to"` key, whose value is the target `Agent` object. This is the most direct way for an agent to decide its next recipient.
2.  **`register_reply`:** You can register reply functions on an agent (or a `GroupChatManager`) that conditionally generate replies or redirect messages based on specific criteria (e.g., message content, sender type).
3.  **GroupChat Manager's `speaker_selection_method`:** For group chats, the manager can use various strategies (`auto`, `round_robin`, `llm_speaker_selection`) to decide who speaks next if no explicit `to` is provided.

In this lesson, we'll focus on creating custom agents that explicitly route messages using the `to` field, demonstrating how agents can take direct control of the conversation flow.


In [ ]:
import autogen
import os

# --- Configuration Setup (2026 Ready) ---
# Ensure your OAI_CONFIG_LIST is set up with modern LLM models.
# Example OAI_CONFIG_LIST content (saved as OAI_CONFIG_LIST in the same directory or environment variable):
# [
#     {
#         "model": "gpt-4o",
#         "api_key": "YOUR_OPENAI_API_KEY"
#     },
#     {
#         "model": "gpt-4-turbo",
#         "api_key": "YOUR_OPENAI_API_KEY"
#     }
# ]

# Fallback for demonstration if OAI_CONFIG_LIST is not found
try:
    config_list = autogen.config_list_from_json(
        "OAI_CONFIG_LIST",
        filter_dict={
            "model": ["gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo"], # Prioritize newer models
        },
    )
except ValueError:
    print("Warning: OAI_CONFIG_LIST not found or invalid. Using dummy config for demonstration.")
    config_list = [
        {
            "model": "dummy-model", # Placeholder for demonstration
            "api_key": "sk-dummy"
        }
    ]

llm_config = {
    "timeout": 60,
    "cache_seed": 42,
    "config_list": config_list,
    "temperature": 0,
}

# --- Custom Agent Definitions ---

class DataAnalystAgent(autogen.ConversableAgent):
    """
    A custom data analyst agent that simulates data processing and explicitly routes
    its findings to a ReportGeneratorAgent.
    """
    def __init__(self, name, llm_config=None, report_generator_agent=None, **kwargs):
        super().__init__(name, llm_config=llm_config, **kwargs)
        self.description = (
            "A data analyst agent that processes data and passes findings to the report generator. "
            "It does not generate reports itself."
        )
        # Store a reference to the next agent in the workflow for explicit routing
        self.report_generator_agent = report_generator_agent

    def generate_reply(self, messages=None, sender=None, exclude=None, **kwargs):
        # Get the last message in the conversation history
        last_message = messages[-1] if messages else {}
        content = last_message.get("content", "")

        # Check if the message is a data analysis request
        if "analyze data" in content.lower() and "q3 sales" in content.lower():
            print(f"\n--- {self.name} is analyzing data from {sender.name} ---")
            # Simulate data analysis process
            analysis_result = (
                "Simulated analysis: Key trend for Q3 sales is a 15% growth, "
                "primarily driven by new product launches in the APAC region. "
                "Customer acquisition costs remained stable. This data is ready for reporting."
            )

            if self.report_generator_agent:
                # Explicitly route the analysis result to the ReportGeneratorAgent
                # The 'to' field in the reply dictionary tells AutoGen where to send this message next.
                print(f"--- {self.name} routing analysis to {self.report_generator_agent.name} ---")
                return {"content": analysis_result, "to": self.report_generator_agent}
            else:
                # Fallback if no report generator is linked
                return {"content": analysis_result + "\n(No report generator found to forward to.)"}
        
        # If this agent cannot handle the request, return False to indicate no reply
        # or let the default LLM reply mechanism take over if desired.
        return False

class ReportGeneratorAgent(autogen.ConversableAgent):
    """
    A custom report generator agent that takes analysis results and formats them
    into a concise report, then routes the final report back to the UserProxyAgent.
    """
    def __init__(self, name, llm_config=None, user_proxy_agent=None, **kwargs):
        super().__init__(name, llm_config=llm_config, **kwargs)
        self.description = (
            "A report generator agent that takes analysis results and formats them "
            "into a concise report. It does not perform data analysis."
        )
        # Store a reference to the user proxy for sending the final report
        self.user_proxy_agent = user_proxy_agent

    def generate_reply(self, messages=None, sender=None, exclude=None, **kwargs):
        last_message = messages[-1] if messages else {}
        content = last_message.get("content", "")

        # Check if the message contains simulated analysis results
        if "Simulated analysis" in content:
            print(f"\n--- {self.name} is generating report from {sender.name} ---")
            # Format the analysis into a report
            report_content = (
                f"**Final Business Report (Q3 Sales):**\n\n"
                f"Based on the detailed analysis: '{content}', we observe a significant "
                f"15% growth in Q3 sales, primarily fueled by successful new product launches "
                f"in the APAC region. Customer acquisition costs remained stable, indicating "
                f"efficient market penetration.\n\n"
                f"**Recommendation:** To sustain this momentum, we recommend a targeted "
                f"marketing campaign focusing on the new product lines and exploring "
                f"expansion opportunities in similar emerging markets. This report is complete."
            )
            
            if self.user_proxy_agent:
                # Explicitly route the final report back to the UserProxyAgent
                print(f"--- {self.name} routing final report to {self.user_proxy_agent.name} ---")
                return {"content": report_content, "to": self.user_proxy_agent}
            else:
                # Fallback if no user proxy is linked
                return {"content": report_content + "\n(No user proxy found to send final report to.)"}
        
        return False

# --- Agent Instantiation and Conversation Setup ---

# 1. User Proxy Agent: Initiates the conversation and receives the final output.
user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER", # Set to "ALWAYS" for human intervention
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: "final report" in x.get("content", "").lower(),
    code_execution_config=False, # No code execution for this example
    llm_config=llm_config, # UserProxyAgent can also use LLM for its replies if needed
    system_message="You are a helpful user proxy. You initiate tasks and receive final reports."
)

# 2. Instantiate custom agents, passing references for explicit routing.
# We need to create the agents in an order that allows passing references.
# Reporter needs user_proxy, Analyst needs reporter.

reporter = ReportGeneratorAgent(
    name="ReportGenerator",
    llm_config=llm_config,
    system_message="You are a report generator. Your job is to take analysis results and format them into a concise report. Do not perform data analysis.",
    user_proxy_agent=user_proxy # Pass reference to the UserProxyAgent for final routing
)

analyst = DataAnalystAgent(
    name="DataAnalyst",
    llm_config=llm_config,
    system_message="You are a data analyst. Your job is to analyze data and pass findings to the report generator. Do not generate reports yourself.",
    report_generator_agent=reporter # Pass reference to the ReportGeneratorAgent for routing
)

# --- Start the Conversation ---

print("\n--- Starting conversation with custom agents and explicit routing ---")
user_proxy.initiate_chat(
    analyst, # The user_proxy starts the conversation by sending a message to the DataAnalystAgent
    message="Please analyze data for Q3 sales performance and provide a summary. Then, generate a comprehensive report based on the findings."
)
print("\n--- Conversation ended ---")


### Interpreting the Output and Use Cases

When you run the code, you'll observe a clear flow of messages, orchestrated by the custom agents' internal logic:

1.  The `User` (UserProxyAgent) initiates the conversation by sending a request to the `DataAnalyst`.
2.  The `DataAnalyst` agent's `generate_reply` method detects the analysis request. It then simulates the analysis and, crucially, returns a reply dictionary with a `"to": ReportGenerator` field. This explicitly tells AutoGen to send the analysis result *directly* to the `ReportGenerator` agent, bypassing the `User` or any default group chat manager.
3.  The `ReportGenerator` agent receives the analysis result. Its `generate_reply` method processes this, formats a final report, and returns a reply with `"to": User`. This directs the completed report back to the `User`.
4.  The `User` (UserProxyAgent) receives the final report and, because its `is_termination_msg` condition is met, terminates the conversation.

This demonstrates a powerful pattern: **agents can dynamically decide the next step in a workflow and explicitly route messages to specific collaborators.**

#### Performance Trade-offs and Considerations:

*   **Development Effort:** Custom agents require more upfront coding to define their specific logic and routing rules compared to simply configuring `AssistantAgent`s.
*   **Debugging Complexity:** As routing becomes more intricate, debugging the flow of messages can be challenging. Clear logging (as shown in the example) and well-defined `generate_reply` methods are crucial.
*   **Flexibility vs. Simplicity:** Custom agents offer unparalleled flexibility and control over agent behavior and conversation flow, but this comes at the cost of increased complexity.
*   **LLM Usage Optimization:** By embedding custom logic, you can reduce unnecessary LLM calls. For instance, the `DataAnalyst` might perform a database query (simulated here) and only involve an LLM if the data needs complex interpretation, or the `ReportGenerator` might use an LLM only for stylistic improvements after structured data processing.
*   **Scalability:** For very large or dynamic agent graphs, managing explicit `to` references can become cumbersome. In such cases, a central `GroupChatManager` with sophisticated `register_reply` functions or a custom `speaker_selection_method` might be more scalable.

#### Typical Use Cases:

*   **Multi-stage Data Pipelines:** An agent extracts data, passes it to another for cleaning, then to a third for analysis, and finally to a fourth for visualization or reporting.
*   **Tool Integration Workflows:** An agent identifies the need for a specific tool (e.g., a search engine, a code interpreter, a specific API), routes the request to a specialized tool-using agent, which then returns the result to the original agent or another for synthesis.
*   **Human-in-the-Loop Systems:** A custom agent can be designed to always route critical decisions or ambiguous situations to a human `UserProxyAgent` for approval or clarification.
*   **Conditional Workflows:** Agents can dynamically choose different sub-workflows based on the input, current state, or external conditions, routing messages to different sets of agents accordingly.
*   **Specialized Knowledge Agents:** An agent might be designed to interface with a specific knowledge base or RAG system, routing queries to it and then processing the retrieved information.


### Resources

*   **AutoGen Documentation:**
    *   [Custom Agents in AutoGen](https://microsoft.github.io/autogen/docs/tutorial/agent_chat#custom-agents)
    *   [ConversableAgent Class Reference](https://microsoft.github.io/autogen/docs/reference/agentchat/conversable_agent)
    *   [Register Reply Function](https://microsoft.github.io/autogen/docs/reference/agentchat/conversable_agent#register_reply)
*   **Advanced AutoGen Examples:** Explore the official AutoGen repository for more complex multi-agent patterns and integrations.
*   **Agentic Workflows:** Understand the broader context of designing and implementing agentic workflows for complex problem-solving.
*   **LLM Orchestration Frameworks:** Research other frameworks and concepts related to orchestrating LLMs and agents (e.g., LangChain, CrewAI) to compare approaches to custom agent development and routing.
